# DDoS detection with XGBoost and SHAP (CSE-CIC-IDS2018)
Starter notebook for the COMP6310 group project. Work through the cells in order.
Everything marked **CONFIRM** depends on the real CSV files, so check it when you open them.
Install once: `pip install pandas numpy scikit-learn xgboost shap matplotlib`

In [8]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)
from xgboost import XGBClassifier
import shap

## 1. Settings (CONFIRM the file names and column names against the real files)

In [14]:
DATA_DIR = Path("../data")                      # put the CSV files here; do not commit them to GitHub
FILES = ["Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
         "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"]
HELD_OUT_TOOL = "HOIC"                       # one of: LOIC-HTTP, LOIC-UDP, HOIC

LABEL, TIME, DST = "Label", "Timestamp", "Dst IP"      # CONFIRM
BASE_FEATURES = [                                       # CONFIRM each name
    "Flow Pkts/s", "Flow Byts/s", "Flow Duration",
    "Flow IAT Mean", "Flow IAT Std", "SYN Flag Cnt",
    "Fwd Pkt Len Mean", "Fwd Pkt Len Std",
]
COUNT_COLS = ["Tot Fwd Pkts", "Tot Bwd Pkts"]   # for the ratio feature
USE_WINDOW_FEATURE = False                       # feature 7 off: Wednesday has no Dst IP
WINDOW = "10s"

## 2. Load the data
The files are large. We read only the columns we need. Some copies of these files contain
repeated header rows, so we remove rows where the label equals "Label" and force numbers.

In [15]:
def load(path):
    head = pd.read_csv(path, nrows=0).columns.tolist()
    print(path.name, "-", len(head), "columns")
    for need in [LABEL] + BASE_FEATURES + COUNT_COLS:
        if need not in head:
            print("   MISSING column:", need)
    keep = [c for c in head if c in set([LABEL, TIME, DST] + BASE_FEATURES + COUNT_COLS)]
    df = pd.read_csv(path, usecols=keep, low_memory=False)
    df = df[df[LABEL] != LABEL].copy()
    for c in BASE_FEATURES + COUNT_COLS:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    if TIME in df: df[TIME] = pd.to_datetime(df[TIME], dayfirst=True, errors="coerce")
    df["source_file"] = path.name
    return df

df = pd.concat([load(DATA_DIR / f) for f in FILES], ignore_index=True)
print(df.shape)
print(df[LABEL].value_counts())
print(df.groupby("source_file")[LABEL].value_counts())
print("Has destination IP:", DST in df, "| has timestamp:", TIME in df)
print(df.groupby("source_file")[[DST, TIME]].apply(lambda g: g.notna().mean()))

Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv - 84 columns
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv - 80 columns
(8997323, 14)
Label
Benign                    7733390
DDOS attack-HOIC           686012
DDoS attacks-LOIC-HTTP     576191
DDOS attack-LOIC-UDP         1730
Name: count, dtype: int64
source_file                                         Label                 
Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv    Benign                    7372557
                                                    DDoS attacks-LOIC-HTTP     576191
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv  DDOS attack-HOIC           686012
                                                    Benign                     360833
                                                    DDOS attack-LOIC-UDP         1730
Name: count, dtype: int64
Has destination IP: True | has timestamp: True
                                                    Dst IP  Timestamp
source_file                                 

## 3. Inspect the labels (answers: do reliable labels exist, how big is each class?)

In [12]:
print(df[LABEL].value_counts())
print(df.groupby("source_file")[LABEL].value_counts())
print("Has destination IP:", DST in df, "| has timestamp:", TIME in df)

Label
Benign                    7733390
DDOS attack-HOIC           686012
DDoS attacks-LOIC-HTTP     576191
DDOS attack-LOIC-UDP         1730
Name: count, dtype: int64
source_file                                         Label                 
Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv    Benign                    7372557
                                                    DDoS attacks-LOIC-HTTP     576191
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv  DDOS attack-HOIC           686012
                                                    Benign                     360833
                                                    DDOS attack-LOIC-UDP         1730
Name: count, dtype: int64
Has destination IP: True | has timestamp: True


## 4. Keep benign and DDoS flows, and name the tool

In [17]:
def tool_of(label):
    s = str(label).lower()
    if "benign" in s: return "Benign"
    if "udp" in s:    return "LOIC-UDP"
    if "hoic" in s:   return "HOIC"
    if "http" in s:   return "LOIC-HTTP"
    return "Other"

df["tool"] = df[LABEL].map(tool_of)
df = df[df["tool"] != "Other"].copy()
df["y"] = (df["tool"] != "Benign").astype(int)       # 1 = DDoS (positive class)
print(df["tool"].value_counts())

tool
Benign       7733390
HOIC          686012
LOIC-HTTP     576191
LOIC-UDP        1730
Name: count, dtype: int64


## 5. Clean (infinite, missing and duplicate values)

In [18]:
n0 = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=BASE_FEATURES + COUNT_COLS)
df = df.drop_duplicates(subset=[c for c in df.columns if c not in ("source_file",)])
print(f"rows: {n0} -> {len(df)}")
print(df["tool"].value_counts())

rows: 8997323 -> 8872017
tool
Benign       7625652
HOIC          668461
LOIC-HTTP     576174
LOIC-UDP        1730
Name: count, dtype: int64


## 6. Build the features (Week 3)
Features 3 and 7 are derived. Identifiers are used only to build feature 7, never as inputs.

In [19]:
df["fwd_bwd_ratio"] = df["Tot Fwd Pkts"] / (df["Tot Bwd Pkts"] + 1)
FEATURES = BASE_FEATURES + ["fwd_bwd_ratio"]

if USE_WINDOW_FEATURE and DST in df and TIME in df and df[TIME].notna().any():
    df["_win"] = df[TIME].dt.floor(WINDOW)
    df["flows_to_dst"] = df.groupby([DST, "_win"])[DST].transform("size")
    FEATURES.append("flows_to_dst")
else:
    print("Window feature skipped (destination IP or timestamp not available).")
print("Model inputs:", FEATURES)

Window feature skipped (destination IP or timestamp not available).
Model inputs: ['Flow Pkts/s', 'Flow Byts/s', 'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'SYN Flag Cnt', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'fwd_bwd_ratio']


In [20]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

def run(held_out, benign_mode="per_day", train_frac=0.7, benign_cap=1_000_000, seed=0):
    d = df[df[TIME].notna()]
    ben = d[d["tool"] == "Benign"]
    att = d[d["tool"] != "Benign"]

    if benign_mode == "per_day":
        # within each day's file, earlier 70% of benign flows -> train, later 30% -> test
        pct = ben.groupby("source_file")[TIME].rank(pct=True, method="first")
        ben_tr, ben_te = ben[pct <= train_frac], ben[pct > train_frac]
    else:  # "by_file": benign from the earlier day -> train, later day -> test
        order = ben.groupby("source_file")[TIME].min().sort_values().index
        ben_tr = ben[ben["source_file"] == order[0]]
        ben_te = ben[ben["source_file"] == order[1]]

    tr = pd.concat([ben_tr.sample(min(len(ben_tr), benign_cap), random_state=seed),
                    att[att["tool"] != held_out]])
    te = pd.concat([ben_te, att[att["tool"] == held_out]])

    spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
    model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                          tree_method="hist", scale_pos_weight=spw,
                          n_jobs=-1, random_state=seed)
    model.fit(tr[FEATURES], tr["y"])

    p = model.predict_proba(te[FEATURES])[:, 1]
    pred, y = p >= 0.5, te["y"].values
    print(f"held out {held_out} | benign split: {benign_mode}")
    print(f"  train: {(tr.y==0).sum():,} benign, {(tr.y==1).sum():,} attack")
    print(f"  test:  {(y==0).sum():,} benign, {(y==1).sum():,} attack")
    print(f"  recall on {held_out}: {pred[y==1].mean():.3f}")
    print(f"  false positive rate on benign: {pred[y==0].mean():.4f}")
    print(f"  PR-AUC: {average_precision_score(y, p):.3f}")
    return model, te

m1, te1 = run("HOIC", "per_day")
m2, te2 = run("HOIC", "by_file")

held out HOIC | benign split: per_day
  train: 1,000,000 benign, 577,904 attack
  test:  2,287,697 benign, 668,461 attack
  recall on HOIC: 0.156
  false positive rate on benign: 0.0070
  PR-AUC: 0.819
held out HOIC | benign split: by_file
  train: 1,000,000 benign, 577,904 attack
  test:  360,808 benign, 668,461 attack
  recall on HOIC: 0.195
  false positive rate on benign: 0.0000
  PR-AUC: 1.000


In [21]:
from sklearn.metrics import roc_auc_score
import numpy as np

for name, m, te in [("per_day", m1, te1), ("by_file", m2, te2)]:
    p = m.predict_proba(te[FEATURES])[:, 1]
    y = te["y"].values
    print(name, "| ROC-AUC:", round(roc_auc_score(y, p), 4))
    print("  HOIC score 10/50/90th pct:", np.percentile(p[y==1], [10, 50, 90]).round(3))
    print("  benign score 50/99/99.9th pct:", np.percentile(p[y==0], [50, 99, 99.9]).round(4))
    for t in [0.5, 0.2, 0.1, 0.05, 0.01]:
        pr = p >= t
        print(f"  threshold {t}: recall {pr[y==1].mean():.3f}, FPR {pr[y==0].mean():.4f}")

print(te2.groupby("tool")[FEATURES].median().T)
print(m2.get_booster().get_score(importance_type="gain"))

per_day | ROC-AUC: 0.963
  HOIC score 10/50/90th pct: [0.    0.    0.611]
  benign score 50/99/99.9th pct: [0.     0.4152 0.9604]
  threshold 0.5: recall 0.156, FPR 0.0070
  threshold 0.2: recall 0.291, FPR 0.0149
  threshold 0.1: recall 0.323, FPR 0.0194
  threshold 0.05: recall 0.386, FPR 0.0220
  threshold 0.01: recall 0.388, FPR 0.0282
by_file | ROC-AUC: 0.9997
  HOIC score 10/50/90th pct: [0.   0.   0.58]
  benign score 50/99/99.9th pct: [0. 0. 0.]
  threshold 0.5: recall 0.195, FPR 0.0000
  threshold 0.2: recall 0.279, FPR 0.0000
  threshold 0.1: recall 0.372, FPR 0.0000
  threshold 0.05: recall 0.373, FPR 0.0000
  threshold 0.01: recall 0.401, FPR 0.0000
tool                     Benign         HOIC
Flow Pkts/s         4778.156997   474.962681
Flow Byts/s       841018.582200     0.000000
Flow Duration       1465.000000  6733.000000
Flow IAT Mean        244.166667  3244.000000
Flow IAT Std         326.072282     0.000000
SYN Flag Cnt           0.000000     0.000000
Fwd Pkt Len Mea

In [22]:
# 1. Profile each attack tool
att = df[df["tool"] != "Benign"]
print((att["Fwd Pkt Len Mean"] == 0).groupby(att["tool"]).mean().rename("share with zero fwd payload"))
print(att.groupby("tool")[FEATURES].median().T.round(1))

# 2. Other held-out direction
m3, te3 = run("LOIC-HTTP", "per_day")

# 3. In-distribution baseline (random split, all tools seen in training)
from sklearn.model_selection import train_test_split
ben = df[df["tool"] == "Benign"].sample(1_000_000, random_state=0)
d = pd.concat([ben, df[df["tool"] != "Benign"]])
tr, te = train_test_split(d, test_size=0.3, stratify=d["tool"], random_state=0)
spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
mb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
                   scale_pos_weight=spw, n_jobs=-1, random_state=0).fit(tr[FEATURES], tr["y"])
pred = mb.predict_proba(te[FEATURES])[:, 1] >= 0.5
print("baseline FPR:", round(pred[te["y"].values == 0].mean(), 4))
for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
    m = (te["tool"] == t).values
    print(f"baseline recall {t}: {pred[m].mean():.3f} (n={m.sum():,})")

tool
HOIC         0.755034
LOIC-HTTP    0.496531
LOIC-UDP     0.000000
Name: share with zero fwd payload, dtype: float64
tool                HOIC  LOIC-HTTP     LOIC-UDP
Flow Pkts/s        475.0        3.8       1016.4
Flow Byts/s          0.0      520.2      32523.7
Flow Duration     6733.0  1750175.0  119777312.0
Flow IAT Mean     3244.0   308419.7        983.9
Flow IAT Std         0.0   151970.6      19526.7
SYN Flag Cnt         0.0        0.0          0.0
Fwd Pkt Len Mean     0.0        6.7         32.0
Fwd Pkt Len Std      0.0       11.5          0.0
fwd_bwd_ratio        2.0        0.6     119758.5
held out LOIC-HTTP | benign split: per_day
  train: 1,000,000 benign, 670,191 attack
  test:  2,287,697 benign, 576,174 attack
  recall on LOIC-HTTP: 0.003
  false positive rate on benign: 0.0054
  PR-AUC: 0.439
baseline FPR: 0.0084
baseline recall HOIC: 0.999 (n=200,539)
baseline recall LOIC-HTTP: 0.987 (n=172,852)
baseline recall LOIC-UDP: 1.000 (n=519)


## 7. Split: hold out one tool
Train on the other two tools. Test on the held-out tool. Benign flows are split by time:
those inside the held-out tool's attack period go to the test set, the rest to training,
so that benign traffic from the test period is not in training.

In [ ]:
is_held = df["tool"] == HELD_OUT_TOOL
if TIME in df and df.loc[is_held, TIME].notna().any():
    t0, t1 = df.loc[is_held, TIME].min(), df.loc[is_held, TIME].max()
    benign_test = (df["tool"] == "Benign") & df[TIME].between(t0, t1)
else:
    print("No timestamps: benign flows split at random (weaker, say so in the report).")
    rng = np.random.default_rng(0)
    benign_test = (df["tool"] == "Benign") & (rng.random(len(df)) < 0.3)

test_mask = is_held | benign_test
train, test = df[~test_mask], df[test_mask]
print("train:", train["y"].value_counts().to_dict(), "| test:", test["y"].value_counts().to_dict())
print("train tools:", train["tool"].value_counts().to_dict())

## 8. Train XGBoost
Class weights handle imbalance inside training only (no SMOTE yet). Tune on a validation split
of the training data, never on the held-out tool.

In [ ]:
Xtr, ytr = train[FEATURES], train["y"]
Xte, yte = test[FEATURES], test["y"]
spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="logloss",
                      n_jobs=-1, random_state=0)
model.fit(Xtr, ytr)

## 9. Evaluate (flag plus confidence score)

In [ ]:
score = model.predict_proba(Xte)[:, 1]            # confidence score for DDoS
def report(score, y, threshold=0.5):
    flag = (score >= threshold).astype(int)       # the flag
    tn, fp, fn, tp = confusion_matrix(y, flag, labels=[0, 1]).ravel()
    return dict(threshold=threshold, recall=recall_score(y, flag), precision=precision_score(y, flag, zero_division=0),
                f1=f1_score(y, flag), false_alarm_rate=fp / max(fp + tn, 1), TP=tp, FP=fp, FN=fn, TN=tn)
print(pd.Series(report(score, yte)))
if yte.nunique() == 2:
    print("ROC AUC:", roc_auc_score(yte, score), "| PR AUC:", average_precision_score(yte, score))
print(pd.DataFrame([report(score, yte, t) for t in [0.1, 0.3, 0.5, 0.7, 0.9]]).round(4))

## 10. Explain with SHAP (Wang et al. idea, applied to a tree model)

In [ ]:
sample = Xte.sample(min(2000, len(Xte)), random_state=0)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(sample)
shap.summary_plot(sv, sample, show=False); plt.tight_layout(); plt.savefig("shap_summary.png", dpi=150); plt.show()

# one flagged flow: why was it flagged?
i = int(np.argmax(model.predict_proba(sample)[:, 1]))
print(pd.Series(sv[i], index=FEATURES).sort_values(key=abs, ascending=False).round(3))

## Next steps
1. Repeat with HELD_OUT_TOOL = "LOIC-HTTP" and "LOIC-UDP" and compare.
2. Check explanation stability: retrain with different random_state values and compare the top SHAP features.
3. Check false alarms on benign flows with high packet rates.
4. Optionally add a Decision Tree baseline or SMOTE inside the training data only.